In [ ]:
# KALMAN DERIVED REBUILD v2.4 — FROZEN IEX CONTRACT / ONE CELL
# Contract proven on legacy 20:
# reconstructed_fixed4_net_return = reconstructed_fixed4_raw_return * weight - cost_proxy
from google.colab import drive
drive.mount("/content/drive",force_remount=False)
from pathlib import Path
from datetime import datetime,timezone
import json,shutil,hashlib
import numpy as np,pandas as pd

R=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results")
ROOT=R/"open_revalidation_v1"; AUD=ROOT/"open_revalidation_trade_audit.parquet"
a=pd.read_parquet(AUD).copy()
price=["entry_price_iex","fixed4_exit_price_iex","prev_close_price_iex","open_0_price_iex","open_5_price_iex","open_15_price_iex"]
for c in price+["weight","cost_proxy"]: a[c]=pd.to_numeric(a[c],errors="coerce")
complete=a[price+["weight","cost_proxy"]].notna().all(axis=1)
old=a["reconstructed_fixed4_net_return"].notna()
old_net=pd.to_numeric(a["reconstructed_fixed4_net_return"],errors="coerce").copy()

def r(x,e): return x/e-1
a.loc[complete,"position_return_prev_close"]=r(a.loc[complete,"prev_close_price_iex"],a.loc[complete,"entry_price_iex"])
a.loc[complete,"position_return_open"]=r(a.loc[complete,"open_0_price_iex"],a.loc[complete,"entry_price_iex"])
a.loc[complete,"position_return_5m"]=r(a.loc[complete,"open_5_price_iex"],a.loc[complete,"entry_price_iex"])
a.loc[complete,"position_return_15m"]=r(a.loc[complete,"open_15_price_iex"],a.loc[complete,"entry_price_iex"])
a.loc[complete,"overnight_gap_return"]=a.loc[complete,"open_0_price_iex"]/a.loc[complete,"prev_close_price_iex"]-1
a.loc[complete,"open_momentum_5m"]=a.loc[complete,"open_5_price_iex"]/a.loc[complete,"open_0_price_iex"]-1
a.loc[complete,"open_momentum_15m"]=a.loc[complete,"open_15_price_iex"]/a.loc[complete,"open_0_price_iex"]-1
a.loc[complete,"giveback_prev_close_to_5m"]=a.loc[complete,"position_return_prev_close"]-a.loc[complete,"position_return_5m"]
a.loc[complete,"reconstructed_fixed4_raw_return"]=r(a.loc[complete,"fixed4_exit_price_iex"],a.loc[complete,"entry_price_iex"])
a.loc[complete,"reconstructed_fixed4_net_return"]=a.loc[complete,"reconstructed_fixed4_raw_return"]*a.loc[complete,"weight"]-a.loc[complete,"cost_proxy"]

# Frozen legacy contract must reproduce all 20 exactly.
err=(pd.to_numeric(a.loc[old,"reconstructed_fixed4_net_return"])-old_net.loc[old]).abs()
print("[LEGACY CONTRACT] n=",int(old.sum()),"median_err=",float(err.median()),"max_err=",float(err.max()))
if len(err)!=20 or err.max()>1e-12: raise RuntimeError("Frozen legacy contract reproduction failed; no write.")

derived=["position_return_prev_close","position_return_open","position_return_5m","position_return_15m","overnight_gap_return","open_momentum_5m","open_momentum_15m","giveback_prev_close_to_5m","reconstructed_fixed4_raw_return","reconstructed_fixed4_net_return"]
print("\n[COMPLETENESS]\n",a[derived].notna().sum().to_string())
before=int(old.sum()); after=int(a["reconstructed_fixed4_net_return"].notna().sum())
if after!=int(complete.sum()) or after!=888: raise RuntimeError(f"Expected 888 reconstructed rows, got {after}; no write.")

# Explicit unresolved row
miss=a.loc[~complete,["fold","symbol","entry_timestamp","exit_timestamp"]+price]
print("\n[UNRESOLVED]\n",miss.to_string(index=False))

stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
bak=ROOT/f"open_revalidation_trade_audit.pre_derived_v2_4_{stamp}.parquet"
tmp=ROOT/"open_revalidation_trade_audit.v2_4.tmp.parquet"
shutil.copy2(AUD,bak); a.to_parquet(tmp,index=False); tmp.replace(AUD)
rep={"schema":"kalman-open-revalidation-derived-v2.4","research_only":True,"production_changed":False,"live_trading":False,"neon_write":False,"contract":"iex_raw_return * weight - cost_proxy","legacy_rows":20,"legacy_max_abs_err":float(err.max()),"rows":len(a),"complete_rows":int(complete.sum()),"derived_before":before,"derived_after":after,"unresolved_rows":int((~complete).sum()),"backup":str(bak),"audit":str(AUD)}
(ROOT/"derived_rebuild_v2_4_report.json").write_text(json.dumps(rep,indent=2))
print("\n[FINAL]\n",json.dumps(rep,indent=2))
print("\nNEXT: do NOT rerun v1.9. Its candidate-vs-baseline contract is obsolete. Run corrected OPEN-policy validation next.")
